In [38]:
#%pip install faiss-cpu
#%pip install langchain
#%pip install langchain_core
#%pip install -U langchain-community
%pip install sentence-transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.1.2-py3-none-any.whl (488 kB)
Note: you may need to restart the kernel to use updated packages.


In [31]:
%pip install langchain==0.3.27

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.1.0
    Uninstalling langchain-core-1.1.0:
      Successfully uninstalled langchain-core-1.1.0
  Attempting uninstall: langchain
    Found existing installation: langchain 1.0.0
    Uninstalling langchain-1.0.0:
      Successfully uninstalled langchain-1.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [51]:
from langchain_core.documents import Document
from langchain.retrievers.parent_document_retriever import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
import langchain
langchain.debug = False
from langchain.vectorstores.faiss import FAISS  # si usas vectorstore FAISS de LangChain
from langchain.embeddings import HuggingFaceEmbeddings  # para usar tu modelo PlanTL-GOB-ES
import os
import pandas as pd

# --- Función para cargar los documentos CANTEMIST como Document de LangChain
def load_cantemist_langchain(base_dir="../data/cantemist"):
    docs = []
    subsets = ["train-set", "dev-set1", "dev-set2", "test-set"]
    for subset in subsets:
        txt_folder = os.path.join(base_dir, subset, "cantemist-coding", "txt")
        if not os.path.isdir(txt_folder):
            continue
        for fname in os.listdir(txt_folder):
            if not fname.endswith(".txt"):
                continue
            path = os.path.join(txt_folder, fname)
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
            metadata = {"source": f"{subset}/{fname}"}
            docs.append(Document(page_content=text, metadata=metadata))
    return docs

# --- Definir splitters
# Parent splitter: divide en pedazos grandes — podrías usar documento completo, o párrafos largos
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=20000,  # tamaño “padre”
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)

# Child splitter: fragmentos pequeños para embedding
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    length_function=len,
    add_start_index=True
)

# --- Embeddings con PlanTL-GOB-ES
embeddings = HuggingFaceEmbeddings(
    model_name="PlanTL-GOB-ES/roberta-base-biomedical-clinical-es",
    # podrías pasar tokenizer, device u otros parámetros si es necesario
)

# --- VectorStore FAISS
# Usamos FAISS desde langchain para indexar los fragmentos “hijos”
vectorstore = FAISS.from_documents(
    documents=child_splitter.split_documents(load_cantemist_langchain()),
    embedding=embeddings
)

# --- Docstore para los padres
from langchain.storage import InMemoryStore
docstore = InMemoryStore()

# --- Construir el ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter, search_kwargs = {'k':10}
)

# --- Agregar los documentos (padres) al retriever
docs = load_cantemist_langchain()
retriever.add_documents(docs)

# --- Ahora podemos hacer consultas
query = "tumor abdominal biopsia"
# Esto va a usar los embeddings de los hijos, pero devolverá textos “padres”
results = retriever.invoke(query)

for doc in results:
    print("--- Padre recuperado ---")
    print("Texto:", doc.page_content[:500], "…")
    print("Metadata:", doc.metadata)
    print("\n")


No sentence-transformers model found with name PlanTL-GOB-ES/roberta-base-biomedical-clinical-es. Creating a new one with mean pooling.
Some weights of RobertaModel were not initialized from the model checkpoint at PlanTL-GOB-ES/roberta-base-biomedical-clinical-es and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--- Padre recuperado ---
Texto: Anamnesis
ANTECEDENTES
Paciente varón de 62 años, caucásico, sin alergias medicamentosas conocidas ni hábitos tóxicos. Vive en su domicilio con su esposa. Tiene 2 hijas que viven en la misma ciudad. Independiente para las actividades básicas de la vida diaria. Performance status: ECOG-0. Como antecedentes patológicos presenta hipertensión arterial primaria en tratamiento farmacológico, diabetes mellitus tipo 2 en tratamiento con antidiabéticos orales, dislipemia en tratamiento farmacológico, car …
Metadata: {'source': 'train-set/cc_onco794.txt', 'start_index': 0}


--- Padre recuperado ---
Texto: Anamnesis

2003
Tras notarse una lesión en la lengua, acudió a su médico de Atención Primaria, por lo que fue derivada al especialista de ORL. Tras su valoración, se decidió extirpación de la lesión, practicándose glosectomía subtotal ampliada a pared lateral izquierda de faringe, vaciamiento cervical radical izquierdo y supraomohioideo derecho, realizándose rec

In [52]:
# --- Ahora podemos hacer consultas

query = "carcinoma"
# Esto va a usar los embeddings de los hijos, pero devolverá textos “padres”
results = retriever.invoke(query)

for doc in results:
    print("--- Padre recuperado ---")
    print("Texto:", doc.page_content[:-1], "…")
    print("Metadata:", doc.metadata)
    print("\n")

--- Padre recuperado ---
Texto: ANAMNESIS
Varón de 67 años que fue remitido en noviembre de 2014 a consultas externas de Medicina Interna desde atención primaria para estudio de lumbalgia. Refería dolor lumbar irradiado a ambos miembros inferiores (de predominio en miembro inferior izquierdo), de 3 meses de evolución, e intensidad creciente, que no se controlaba con analgesia habitual. En ese momento, en tratamiento con fentanilo transdérmico (FTD) 25 mcg. 
Como antecedentes, destaca dislipemia en tratamiento con estatinas. Exfumador de 1 paquete al día durante 25 años (IPA 25). Jubilado reciente, con buen apoyo familiar, Índice de Karnofsky de 100.

EXPLORACIÓN FÍSICA
Performance status (PS) ECOG 0.
Auscultación cardiaca rítmica y sin soplos audibles. Auscultación pulmonar con murmullo vesicular conservado, sin ruidos patológicos. Resto de la exploración dentro de la normalidad.

PRUEBAS COMPLEMENTARIAS
1. Rx lumbar y sacra: Hiperlordosis lumbar con cambios degenerativos a nivel de L5